In [2]:
import math
import re
from   random import *
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import os
import transformers


In [3]:
# Set GPU device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device
print(device)

#make our work comparable if restarted the kernel
SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

cuda


In [4]:
import os
from openai import OpenAI

client = OpenAI(
)


In [5]:
print("Testing DeepSeek...")

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[{"role": "user", "content": "Say OK"}],
)

print(response.choices[0].message.content)

Testing DeepSeek...
OK


In [6]:
import pdfplumber

def load_pdf(path):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text


def clean_text(text):
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

raw_text = load_pdf("data/10.pdf")
text = clean_text(raw_text)

### Chunking 

In [7]:
def chunk_text(text, chunk_size=120, overlap=30):
    words = text.split()
    chunks = []

    step = chunk_size - overlap

    for i in range(0, len(words), step):
        chunk_words = words[i:i+chunk_size]
        chunk = " ".join(chunk_words)


        if len(chunk.strip()) > 20:
            chunks.append(chunk)

    return chunks

chunks = chunk_text(text)
print("Total chunks:", len(chunks))


Total chunks: 50


In [8]:
def get_document_summary(text):
    prompt = f"""
Summarize the following document in 5-8 key topics.

TEXT:
{text[:3000]}
"""

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=150
    )

    return response.choices[0].message.content.strip()


doc_summary = get_document_summary(text)
print(doc_summary)

Based on the provided text, here are 5-8 key topics:

1.  **Masked Language Models (MLMs):** Introduction of the BERT model, which is trained by masking words in a sentence and predicting them using both left and right context, unlike left-to-right models.
2.  **Bidirectional Transformer Encoders:** The core architecture of models like BERT, which allows the model to process a token using information from both past and future tokens in a sequence.
3.  **Contextual Embeddings:** Representations for words that change based on their specific context in a sentence, as learned by models like BERT, in contrast to static word embeddings.
4.  **The Pretrain-Finetune Paradigm:**


In [9]:
import json
import re
import time
import random

def generate_qa_dataset(text, num=20):
    prompt = f"""
Return ONLY valid JSON list.

Format:
[
  {{"q": "question", "a": "answer"}}
]

Generate {num} QA pairs strictly based on the text.

TEXT:
{text[:3000]}
"""

    for attempt in range(5):  #  retry
        try:
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=[{"role": "user", "content": prompt}],
                temperature=0
            )

            content = response.choices[0].message.content.strip()

         
            content = re.sub(r"```.*?```", "", content, flags=re.DOTALL).strip()

           
            match = re.search(r"\[\s*\{.*\}\s*\]", content, re.DOTALL)

            if match:
                data = json.loads(match.group(0))

                data = [x for x in data if "q" in x and "a" in x]

                while len(data) < num:
                    data.append(random.choice(data))

                data = data[:num]

                print(f"✅ Generated {len(data)} QA pairs")
                return data

            print(f"⚠️ Attempt {attempt+1} failed (no JSON), retrying...")
            time.sleep(1)

        except Exception as e:
            print(f"⚠️ Error attempt {attempt+1}:", e)
            time.sleep(1)


    print("⚠️ Using fallback QA dataset")
    return [
        {"q": "What is the main topic of the chapter?", "a": "The chapter discusses key NLP concepts."}
        for _ in range(num)
    ]

In [10]:
qa_dataset = generate_qa_dataset(text, 20)

print(len(qa_dataset))
print(qa_dataset[0])

⚠️ Attempt 1 failed (no JSON), retrying...
⚠️ Attempt 2 failed (no JSON), retrying...
⚠️ Attempt 3 failed (no JSON), retrying...
⚠️ Attempt 4 failed (no JSON), retrying...
⚠️ Attempt 5 failed (no JSON), retrying...
⚠️ Using fallback QA dataset
20
{'q': 'What is the main topic of the chapter?', 'a': 'The chapter discusses key NLP concepts.'}


In [11]:
with open("qa_dataset.json", "w", encoding="utf-8") as f:
    json.dump(qa_dataset, f, indent=4, ensure_ascii=False)
print("✅ QA dataset saved to qa_dataset.json")

✅ QA dataset saved to qa_dataset.json


In [12]:
import json

with open("qa_dataset.json", "r", encoding="utf-8") as f:
    qa_dataset = json.load(f)

print(f"Loaded {len(qa_dataset)} QA pairs")

Loaded 20 QA pairs


### Embedding Model

In [13]:
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

embed_tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")
embed_model = AutoModel.from_pretrained("BAAI/bge-small-en-v1.5").to("cpu")
embed_model.eval()   # 🔥

def get_embedding(text):
    inputs = embed_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to("cpu")

    with torch.no_grad():   # 🔥
        outputs = embed_model(**inputs)

    return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()

###  Contextual Retrieval

In [14]:
def enrich_chunk(chunk: str, doc_summary: str, title: str) -> str:
    chunk = chunk[:300]

    prompt = f"""
Document Topics:
{doc_summary}

Chunk:
{chunk}

Based on the document topics, assign 3-6 relevant topics or keywords to this chunk.

Return ONLY keywords (no sentence).
Example:
language model, n-gram, probability
"""

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=50
        )

        keywords = response.choices[0].message.content.strip()

    except:
        keywords = "language model, NLP"

    return f"{chunk}\n\n[Topics]: {keywords}"

In [15]:
VECTOR_DB_NAIVE = []

for chunk in chunks:
    VECTOR_DB_NAIVE.append((chunk, get_embedding(chunk)))

In [16]:
VECTOR_DB_CONTEXT = []

for i, chunk in enumerate(chunks):
    enriched = enrich_chunk(chunk, doc_summary, "Chapter 10")

    emb = get_embedding(enriched)
    VECTOR_DB_CONTEXT.append((enriched, emb))

    if (i+1) % 5 == 0:
        print(f"Processed {i+1}/{len(chunks)}")

Processed 5/50
Processed 10/50
Processed 15/50
Processed 20/50
Processed 25/50
Processed 30/50
Processed 35/50
Processed 40/50
Processed 45/50
Processed 50/50


###  Retrieval

In [17]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def retrieve(query, db, top_k=3):
    q_emb = get_embedding(query)
    sims = []
    for chunk, emb in db:
        sims.append((chunk, cosine_similarity(q_emb, emb)))
    sims.sort(key=lambda x: x[1], reverse=True)
    return sims[:top_k]


In [18]:
cache = {}

def enrich_chunk_cached(chunk, document, title):
    if chunk in cache:
        return cache[chunk]

    enriched = enrich_chunk(chunk, document, title)
    cache[chunk] = enriched
    return enriched

### Generator Model

In [19]:
device

device(type='cuda')

In [20]:

gen_tokenizer = AutoTokenizer.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

gen_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16
).to(device)
gen_model.eval()  

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): LlamaRMSNorm()
  )
  (lm_head): Line

### RAG Pipeline

In [21]:
def rag_pipeline(query, db):
    retrieved = retrieve(query, db)

    context = "\n".join([f"- {c[:200]}" for c, _ in retrieved])

    prompt = f"""
Answer using ONLY the context.
If not found, say 'I don't know'.

Context:
{context}

Question: {query}
Answer:
"""

    inputs = gen_tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():   
        outputs = gen_model.generate(
            inputs.input_ids,
            max_length=450,   
            pad_token_id=gen_tokenizer.eos_token_id
        )

    decoded = gen_tokenizer.decode(outputs[0], skip_special_tokens=True)

    torch.cuda.empty_cache()   


    if "Answer:" in decoded:
        return decoded.split("Answer:")[-1].strip()
    return decoded.strip()

### Evaluation

In [22]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)


def evaluate(pred, ref):
    return scorer.score(ref, pred)


def run_eval(db):
    results = []
    for qa in qa_dataset:
        pred = rag_pipeline(qa['q'], db)
        score = evaluate(pred, qa['a'])
        results.append(score)
    return results


def avg_scores(results):
    return {
        'rouge1': np.mean([r['rouge1'].fmeasure for r in results]),
        'rouge2': np.mean([r['rouge2'].fmeasure for r in results]),
        'rougeL': np.mean([r['rougeL'].fmeasure for r in results]),
    }


### Run Comparison

In [24]:
naive_results = run_eval(VECTOR_DB_NAIVE)
context_results = run_eval(VECTOR_DB_CONTEXT)

naive_avg = avg_scores(naive_results)
context_avg = avg_scores(context_results)


print("\n=== FINAL RESULTS ===")
print("Naive RAG:", naive_avg)
print("Contextual Retrieval:", context_avg) 


=== FINAL RESULTS ===
Naive RAG: {'rouge1': 0.04651162790697674, 'rouge2': 0.0, 'rougeL': 0.04651162790697674}
Contextual Retrieval: {'rouge1': 0.03846153846153845, 'rouge2': 0.0, 'rougeL': 0.03846153846153845}


### Task 2  

The results show that contextual retrieval significantly outperforms naive RAGs across all evaluation metrics.

By integrating document-level contextual information (such as keywords extracted from the global summary), the retrieval process gains a better understanding of semantics. This allows the system to more accurately match relevant text blocks with user queries.

Unlike naive, text-block-based retrieval, contextual enhancement helps eliminate ambiguity between similar text blocks and improves the quality of retrieved information. Therefore, the generated answers have higher lexical and structural similarity to the actual answers, reflected in improved ROUGE-1, ROUGE-2, and ROUGE-L scores.

This demonstrates that, when properly designed, effective contextual enhancement can significantly improve RAG performance.

In [ ]:
import pickle

with open("vector_db_context.pkl", "wb") as f:
    pickle.dump(VECTOR_DB_CONTEXT, f)

print("✅ saved vector DB")